# Sentiment Analysis on Amazon Reviews

This notebook trains a **binary sentiment classifier** where:

- **1 = Positive**
- **0 = Negative**

We’ll keep the **same model setup** (TF‑IDF + classic ML models) so the accuracy stays the same — but the code is written in a more **beginner‑friendly** way.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn import metrics
import warnings
warnings.filterwarnings("ignore")

1) Dataset Loading---Sajeeb Start

Recommended: put your CSV in the same folder as this notebook and set DATA_PATH.

The CSV must have:

a text column named Review
a label column named Sentiment (0/1)

In [ ]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/NafisAziz/Sentiment-Analysis-on-Amazon-Product-Reviews/refs/heads/main/AmazonReview%20-%20AmazonReview_sentiment01.csv")
data.head()

In [ ]:
data.shape

2) Clean the labels (Sentiment) --> Shihab Start

In [ ]:
# Keep only rows with the columns we need
data = data.dropna(subset=["Review", "Sentiment"]).copy()

# Map common text labels to numbers
label_map = {
    "negative": 0, "neg": 0,
    "positive": 1, "pos": 1,
    "0": 0, "1": 1
}

# Convert to strings first, then replace, then convert to numeric
s = data["Sentiment"].astype(str).str.strip().str.lower()
s = s.replace(label_map)

data["Sentiment"] = pd.to_numeric(s, errors="coerce")

# Keep only valid labels (0 or 1)
data = data[data["Sentiment"].isin([0, 1])].copy()
data["Sentiment"] = data["Sentiment"].astype(int)

data["Sentiment"].value_counts()


In [ ]:
# 3) Light text cleaning: remove stopwords

# NLTK's English stopwords list (embedded so you don't need nltk)
STOP_WORDS = {
'i','me','my','myself','we','our','ours','ourselves','you',"you're","you've","you'll","you'd",'your','yours',
'yourself','yourselves','he','him','his','himself','she',"she's",'her','hers','herself','it',"it's",'its',
'itself','they','them','their','theirs','themselves','what','which','who','whom','this','that',"that'll",
'these','those','am','is','are','was','were','be','been','being','have','has','had','having','do','does','did',
'doing','a','an','the','and','but','if','or','because','as','until','while','of','at','by','for','with','about',
'against','between','into','through','during','before','after','above','below','to','from','up','down','in','out',
'on','off','over','under','again','further','then','once','here','there','when','where','why','how','all','any',
'both','each','few','more','most','other','some','such','only','own','same','so','than','too',
'very','s','t','can','will','just','don','should',"should've",'now','d','ll','m','o','re','ve','y','ain',
'aren',"aren't",'couldn',"couldn't",'didn',"didn't",'doesn','hadn',"hadn't",'hasn',"hasn't",'haven'
'isn','ma','mightn',"mightn't",'mustn',"mustn't",'needn',"needn't",'shan',"shan't",'shouldn',
"shouldn't",'wasn','weren','won',"won't",'wouldn',"wouldn't"
}

def remove_stopwords(text: str) -> str:
    """Remove common English stopwords (very light cleaning)."""
    words = str(text).split()
    return " ".join(w for w in words if w.lower() not in STOP_WORDS)

# Clean the reviews (keep original + cleaned columns)
data["Review_clean"] = data["Review"].apply(remove_stopwords)

data[["Review", "Review_clean", "Sentiment"]].head()

# shihab end

4) Train / test split----Tripty's part

We keep the same split settings:
- **25% test,75% train**
- `random_state=42`
- `stratify=y` (keeps the class balance)


In [ ]:
X = data["Review_clean"].astype(str)
y = data["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size: ", len(X_test))

5) TF‑IDF + Models  ---->Nafis's Part

TF‑IDF turns text into numbers.

We use unigrams + bigrams (1-word and 2-word phrases)
We keep up to 50,000 features (a strong baseline)
Then we compare a few models and pick the best using cross‑validation.

In [ ]:
# TF‑IDF settings
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    lowercase=True
)

# Models to compare (same as before)
models = {
    "LogisticRegression": LogisticRegression(max_iter=4000, class_weight="balanced"),
    "LinearSVC (Calibrated)": CalibratedClassifierCV(
        estimator=LinearSVC(class_weight="balanced"), cv=3
    ),
    "SGD (log_loss)": SGDClassifier(loss="log_loss", max_iter=4000, class_weight="balanced"),
    "MultinomialNB": MultinomialNB()
}


In [ ]:
# 5-fold stratified cross-validation (on the training set only)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, clf in models.items():
    pipe = Pipeline([
        ("tfidf", tfidf),
        ("clf", clf)
    ])

    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    results.append((name, scores.mean(), scores.std()))

    print(f"{name:22s}  mean={scores.mean():.4f}  std={scores.std():.4f}")

# Best = highest CV mean accuracy
best_name, best_mean, best_std = sorted(results, key=lambda x: x[1], reverse=True)[0]
print("\n✅ Best model:", best_name, "| CV mean accuracy:", round(best_mean, 4))
#Nafis End


 6) Train the best model and evaluate on the test set ---->Nafis's Part

In [ ]:
#6) Train the best model and evaluate on the test set

best_model = models[best_name]

best_pipe = Pipeline([
    ("tfidf", tfidf),
    ("clf", best_model)
])

best_pipe.fit(X_train, y_train)

pred = best_pipe.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, pred))
print("\nClassification Report:\n", classification_report(y_test, pred))


7)system model Visualization--->(Asura's Part)

In [ ]:


# Confusion matrix (how many positives/negatives were correct)
cm = confusion_matrix(y_test, pred)

cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[0, 1]
)

cm_display.plot()
plt.show()


8) Trying our own text

In [ ]:
#8) Trying our own text

def predict_sentiment(text: str) -> str:
    cleaned = remove_stopwords(text)
    pred = best_pipe.predict([cleaned])[0]
    return "Positive " if pred == 1 else "Negative "


while True:
    user_text = input("\nEnter a review (type 'exit' to stop): ")

    if user_text.lower() == "exit":
        break

    print("Predicted Sentiment:", predict_sentiment(user_text))